# ReDeeM peptide foundation-model validation report

This report evaluates the current unified peptide foundation model on the **sequence-disjoint validation partition**. It combines:

- source-resolved RT and CCS regression benchmarking;
- RT coordinate-system / source-scale diagnostics;
- forward MS2 intensity prediction and mirror plots;
- spectrum-to-peptide candidate generation and reranking;
- the accepted **v0.13.23 two-view bidirectional MITM** proposal architecture;
- optional **v0.14.0 setwise reranker** results when its validation summary is available;
- source/family stratification and representative spectrum-to-peptide assignments.

**The TEST partition is intentionally not used in this report.**


In [ ]:
from pathlib import Path
import json, math, os, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=RuntimeWarning)
REPORT_DIR = Path(os.environ.get("REDEEM_REPORT_DIR", ".")).resolve()
MANIFEST = REPORT_DIR / "report_manifest.json"
meta = json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}

def path(name, default):
    configured = Path(meta.get(name, REPORT_DIR / default))
    local = REPORT_DIR / default
    if configured.exists():
        return configured
    if local.exists():
        return local
    return configured

forward_path = path("forward_predictions", "forward_predictions.tsv")
fragments_path = path("ms2_fragments", "ms2_fragments.tsv")
source_summary_path = path("forward_source_summary", "forward_source_summary.tsv")
baseline_inverse_path = path("inverse_baseline", "inverse_baseline.tsv")
unified_inverse_path = path("inverse_unified", "inverse_unified.tsv")
spectra_path = path("inverse_unified_spectra", "inverse_unified.spectra.tsv")
provenance_path = path("provenance", "redeem_foundation_expanded_v0133_provenance.tsv")

RT_HARMONIZATION_DIR = Path(
    os.environ.get(
        "REDEEM_RT_HARMONIZATION_DIR",
        "/home/sing/Documents/datasets/redeem_foundation/rt_harmonization_v0136_audit",
    )
).resolve()
RT_HARMONIZATION_CALIBRATION = Path(
    os.environ.get(
        "REDEEM_RT_HARMONIZATION_CALIBRATION_TSV",
        "/home/sing/Documents/transition_lists/redeem_foundation_expanded_v0136_rt_harmonization.tsv",
    )
).resolve()

rt_harmonization_available = all(
    (RT_HARMONIZATION_DIR / name).exists()
    for name in [
        "initial_rt.predictions.tsv",
        "final_rt.predictions.tsv",
        "initial_rt.summary.tsv",
        "final_rt.summary.tsv",
    ]
)

for p in [forward_path, fragments_path, source_summary_path, baseline_inverse_path, unified_inverse_path, spectra_path]:
    if not p.exists():
        raise FileNotFoundError(p)

forward = pd.read_csv(forward_path, sep="\t", na_values=["NA", ""])
fragments = pd.read_csv(fragments_path, sep="\t", na_values=["NA", ""])
source_summary = pd.read_csv(source_summary_path, sep="\t", na_values=["NA", ""])
baseline_inv = pd.read_csv(baseline_inverse_path, sep="\t", na_values=["NA", ""])
unified_inv = pd.read_csv(unified_inverse_path, sep="\t", na_values=["NA", ""])
spectra = pd.read_csv(spectra_path, sep="\t", na_values=["NA", ""])
provenance = pd.read_csv(provenance_path, sep="\t", comment="#") if provenance_path.exists() else pd.DataFrame(columns=["record_index","source_id"])

if rt_harmonization_available:
    rt_harm_initial = pd.read_csv(RT_HARMONIZATION_DIR / "initial_rt.predictions.tsv", sep="\t")
    rt_harm_final = pd.read_csv(RT_HARMONIZATION_DIR / "final_rt.predictions.tsv", sep="\t")
    rt_harm_initial_summary = pd.read_csv(RT_HARMONIZATION_DIR / "initial_rt.summary.tsv", sep="\t")
    rt_harm_final_summary = pd.read_csv(RT_HARMONIZATION_DIR / "final_rt.summary.tsv", sep="\t")
    rt_harm_calibration = (
        pd.read_csv(RT_HARMONIZATION_CALIBRATION, sep="\t", comment="#")
        if RT_HARMONIZATION_CALIBRATION.exists()
        else pd.DataFrame()
    )
else:
    rt_harm_initial = pd.DataFrame()
    rt_harm_final = pd.DataFrame()
    rt_harm_initial_summary = pd.DataFrame()
    rt_harm_final_summary = pd.DataFrame()
    rt_harm_calibration = pd.DataFrame()

MS2_SHAPE_AUDIT_DIR = Path(
    os.environ.get(
        "REDEEM_MS2_SHAPE_AUDIT_DIR",
        "/home/sing/Documents/datasets/redeem_foundation/ms2_shape_v0137_audit",
    )
).resolve()
ms2_shape_available = all(
    (MS2_SHAPE_AUDIT_DIR / name).exists()
    for name in [
        "ms2_shape_records.tsv",
        "ms2_shape_fragments.tsv",
        "ms2_shape_source_summary.tsv",
        "ms2_shape_family_summary.tsv",
        "ms2_shape_paired_summary.tsv",
    ]
)
if ms2_shape_available:
    ms2_shape_records = pd.read_csv(MS2_SHAPE_AUDIT_DIR / "ms2_shape_records.tsv", sep="\t", na_values=["NA", ""])
    ms2_shape_fragments = pd.read_csv(MS2_SHAPE_AUDIT_DIR / "ms2_shape_fragments.tsv", sep="\t", na_values=["NA", ""])
    ms2_shape_source_summary = pd.read_csv(MS2_SHAPE_AUDIT_DIR / "ms2_shape_source_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_shape_family_summary = pd.read_csv(MS2_SHAPE_AUDIT_DIR / "ms2_shape_family_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_shape_paired_summary = pd.read_csv(MS2_SHAPE_AUDIT_DIR / "ms2_shape_paired_summary.tsv", sep="\t", na_values=["NA", ""])
else:
    ms2_shape_records = pd.DataFrame()
    ms2_shape_fragments = pd.DataFrame()
    ms2_shape_source_summary = pd.DataFrame()
    ms2_shape_family_summary = pd.DataFrame()
    ms2_shape_paired_summary = pd.DataFrame()

MS2_ACTIVATION_AUDIT_DIR = Path(
    os.environ.get(
        "REDEEM_MS2_ACTIVATION_AUDIT_DIR",
        "/home/sing/Documents/datasets/redeem_foundation/ms2_activation_v0138_audit",
    )
).resolve()
ms2_activation_available = all(
    (MS2_ACTIVATION_AUDIT_DIR / name).exists()
    for name in [
        "ms2_activation_records.tsv",
        "ms2_activation_fragments.tsv",
        "ms2_activation_source_summary.tsv",
        "ms2_activation_family_summary.tsv",
        "ms2_activation_channel_summary.tsv",
        "ms2_activation_paired_summary.tsv",
    ]
)
if ms2_activation_available:
    ms2_activation_records = pd.read_csv(MS2_ACTIVATION_AUDIT_DIR / "ms2_activation_records.tsv", sep="\t", na_values=["NA", ""])
    ms2_activation_fragments = pd.read_csv(MS2_ACTIVATION_AUDIT_DIR / "ms2_activation_fragments.tsv", sep="\t", na_values=["NA", ""])
    ms2_activation_source_summary = pd.read_csv(MS2_ACTIVATION_AUDIT_DIR / "ms2_activation_source_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_activation_family_summary = pd.read_csv(MS2_ACTIVATION_AUDIT_DIR / "ms2_activation_family_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_activation_channel_summary = pd.read_csv(MS2_ACTIVATION_AUDIT_DIR / "ms2_activation_channel_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_activation_paired_summary = pd.read_csv(MS2_ACTIVATION_AUDIT_DIR / "ms2_activation_paired_summary.tsv", sep="\t", na_values=["NA", ""])
else:
    ms2_activation_records = pd.DataFrame()
    ms2_activation_fragments = pd.DataFrame()
    ms2_activation_source_summary = pd.DataFrame()
    ms2_activation_family_summary = pd.DataFrame()
    ms2_activation_channel_summary = pd.DataFrame()
    ms2_activation_paired_summary = pd.DataFrame()

MS2_B2_RESCUE_AUDIT_DIR = Path(
    os.environ.get(
        "REDEEM_MS2_B2_RESCUE_AUDIT_DIR",
        "/home/sing/Documents/datasets/redeem_foundation/ms2_b2_rescue_v0139_audit",
    )
).resolve()
ms2_b2_rescue_available = all(
    (MS2_B2_RESCUE_AUDIT_DIR / name).exists()
    for name in [
        "ms2_b2_rescue_records.tsv",
        "ms2_b2_rescue_fragments.tsv",
        "ms2_b2_rescue_source_summary.tsv",
        "ms2_b2_rescue_family_summary.tsv",
        "ms2_b2_rescue_channel_summary.tsv",
        "ms2_b2_rescue_head_summary.tsv",
        "ms2_b2_rescue_paired_summary.tsv",
    ]
)
if ms2_b2_rescue_available:
    ms2_b2_rescue_records = pd.read_csv(MS2_B2_RESCUE_AUDIT_DIR / "ms2_b2_rescue_records.tsv", sep="\t", na_values=["NA", ""])
    ms2_b2_rescue_fragments = pd.read_csv(MS2_B2_RESCUE_AUDIT_DIR / "ms2_b2_rescue_fragments.tsv", sep="\t", na_values=["NA", ""])
    ms2_b2_rescue_source_summary = pd.read_csv(MS2_B2_RESCUE_AUDIT_DIR / "ms2_b2_rescue_source_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_b2_rescue_family_summary = pd.read_csv(MS2_B2_RESCUE_AUDIT_DIR / "ms2_b2_rescue_family_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_b2_rescue_channel_summary = pd.read_csv(MS2_B2_RESCUE_AUDIT_DIR / "ms2_b2_rescue_channel_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_b2_rescue_head_summary = pd.read_csv(MS2_B2_RESCUE_AUDIT_DIR / "ms2_b2_rescue_head_summary.tsv", sep="\t", na_values=["NA", ""])
    ms2_b2_rescue_paired_summary = pd.read_csv(MS2_B2_RESCUE_AUDIT_DIR / "ms2_b2_rescue_paired_summary.tsv", sep="\t", na_values=["NA", ""])
else:
    ms2_b2_rescue_records = pd.DataFrame()
    ms2_b2_rescue_fragments = pd.DataFrame()
    ms2_b2_rescue_source_summary = pd.DataFrame()
    ms2_b2_rescue_family_summary = pd.DataFrame()
    ms2_b2_rescue_channel_summary = pd.DataFrame()
    ms2_b2_rescue_head_summary = pd.DataFrame()
    ms2_b2_rescue_paired_summary = pd.DataFrame()

for df in [baseline_inv, unified_inv]:
    for col in ["mass_valid", "from_diffusion", "from_causal_beam", "peptidoform_exact", "sequence_exact", "il_sequence_exact"]:
        if col in df:
            df[col] = df[col].astype(str).str.lower().map({"true": True, "false": False})

source_to_family = {}
for source in forward.source_id.dropna().unique():
    if source.startswith("dphlv2_"):
        family = "DPHLv2"
    elif source.startswith("pxd058337_"):
        family = "PXD058337"
    elif source.startswith("pxd034128_"):
        family = "PXD034128"
    elif source == "ip2_bruker_human":
        family = "IP2/Bruker"
    elif source == "openswath_finetuning":
        family = "OpenSWATH"
    elif source == "pan_human_library":
        family = "Pan-Human"
    elif source == "pxd035249_csf":
        family = "PXD035249"
    elif source == "pride_human_msp":
        family = "PRIDE Human MSP"
    else:
        family = source
    source_to_family[source] = family

forward["family"] = forward.source_id.map(source_to_family)
fragments["family"] = fragments.source_id.map(source_to_family)
if len(provenance):
    provenance["family"] = provenance.source_id.map(source_to_family)

print(f"Report directory: {REPORT_DIR}")
print(f"Forward prediction rows: {len(forward):,}")
print(f"Annotated fragment rows: {len(fragments):,}")
print(f"Baseline inverse candidate rows: {len(baseline_inv):,}")
print(f"Unified inverse candidate rows: {len(unified_inv):,}")
print(f"Unified observed peak rows: {len(spectra):,}")
print(f"v0.13.7 MS2 shape audit available: {ms2_shape_available}")

## 1. Forward-property benchmark overview

In [ ]:
def regression_metrics(df, target, pred):
    d = df[[target, pred]].dropna()
    if len(d) == 0:
        return {"n": 0}
    y = d[target].to_numpy(float)
    p = d[pred].to_numpy(float)
    err = p-y
    pearson = stats.pearsonr(y,p).statistic if len(d) > 1 and np.std(y)>0 and np.std(p)>0 else np.nan
    spearman = stats.spearmanr(y,p).statistic if len(d) > 1 else np.nan
    sst = np.square(y-y.mean()).sum()
    r2 = 1-np.square(err).sum()/sst if sst>0 else np.nan
    return {
        "n": len(d), "MAE": np.mean(np.abs(err)), "RMSE": np.sqrt(np.mean(err**2)),
        "Pearson r": pearson, "Spearman rho": spearman, "R2": r2,
    }

rows=[]
for checkpoint, d in forward.groupby("checkpoint"):
    for task, target, pred in [("RT","target_rt","predicted_rt"),("CCS","target_ccs","predicted_ccs")]:
        m=regression_metrics(d,target,pred)
        rows.append({"checkpoint":checkpoint,"task":task,**m})
summary_global=pd.DataFrame(rows)
display(summary_global.round(4))

ms2_global=(forward.dropna(subset=["ms2_cosine"])
            .groupby("checkpoint")
            .agg(ms2_records=("record_index","size"),
                 mean_cosine=("ms2_cosine","mean"),
                 median_cosine=("ms2_cosine","median"),
                 mean_spectral_angle=("ms2_spectral_angle","mean"),
                 median_spectral_angle=("ms2_spectral_angle","median"),
                 mean_pearson=("ms2_pearson","mean")))
display(ms2_global.round(4))

In [ ]:
def scatter_plot(df, target, pred, title, xlabel, ylabel):
    d=df[[target,pred]].dropna()
    fig, ax = plt.subplots(figsize=(7,6))
    ax.scatter(d[target], d[pred], s=8, alpha=0.25)
    lo=min(d[target].min(),d[pred].min()); hi=max(d[target].max(),d[pred].max())
    ax.plot([lo,hi],[lo,hi], linestyle="--")
    if len(d)>2:
        slope, intercept=np.polyfit(d[target],d[pred],1)
        xs=np.linspace(lo,hi,100)
        ax.plot(xs, intercept+slope*xs)
    m=regression_metrics(d,target,pred)
    ax.set_title(title+f"\nN={m['n']:,} | r={m.get('Pearson r',np.nan):.3f} | RMSE={m.get('RMSE',np.nan):.3f}")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    fig.tight_layout(); plt.show()

for checkpoint,d in forward.groupby("checkpoint"):
    scatter_plot(d,"target_rt","predicted_rt",f"RT prediction — {checkpoint}","Observed normalized RT","Predicted normalized RT")

for checkpoint,d in forward.groupby("checkpoint"):
    scatter_plot(d,"target_ccs","predicted_ccs",f"CCS prediction — {checkpoint}","Observed CCS","Predicted CCS")

In [ ]:
def residual_plot(df, target, pred, title):
    d=df[[target,pred]].dropna().copy()
    d["residual"]=d[pred]-d[target]
    fig, ax=plt.subplots(figsize=(7,5))
    ax.scatter(d[target],d.residual,s=8,alpha=0.25)
    ax.axhline(0,linestyle="--")
    ax.set_xlabel("Observed")
    ax.set_ylabel("Prediction - observed")
    ax.set_title(title)
    fig.tight_layout(); plt.show()

for checkpoint,d in forward.groupby("checkpoint"):
    residual_plot(d,"target_rt","predicted_rt",f"RT residuals — {checkpoint}")
for checkpoint,d in forward.groupby("checkpoint"):
    residual_plot(d,"target_ccs","predicted_ccs",f"CCS residuals — {checkpoint}")

## 2. Source- and family-resolved RT/CCS performance

In [ ]:
rows=[]
for (checkpoint,source),d in forward.groupby(["checkpoint","source_id"]):
    for task,target,pred in [("RT","target_rt","predicted_rt"),("CCS","target_ccs","predicted_ccs")]:
        m=regression_metrics(d,target,pred)
        rows.append({"checkpoint":checkpoint,"source_id":source,"family":source_to_family.get(source,source),"task":task,**m})
source_metrics=pd.DataFrame(rows)
display(source_metrics.sort_values(["task","checkpoint","RMSE"],na_position="last").round(4))

for task in ["RT","CCS"]:
    d=source_metrics[(source_metrics.task==task)&(source_metrics.n>=8)].copy()
    if len(d):
        pivot=d.pivot(index="source_id",columns="checkpoint",values="RMSE").sort_index()
        fig,ax=plt.subplots(figsize=(10,max(5,0.35*len(pivot))))
        pivot.plot.barh(ax=ax)
        ax.set_title(f"{task} RMSE by source")
        ax.set_xlabel("RMSE")
        fig.tight_layout(); plt.show()

In [ ]:
family_rows=[]
for (checkpoint,family),d in forward.groupby(["checkpoint","family"]):
    for task,target,pred in [("RT","target_rt","predicted_rt"),("CCS","target_ccs","predicted_ccs")]:
        family_rows.append({"checkpoint":checkpoint,"family":family,"task":task,**regression_metrics(d,target,pred)})
family_metrics=pd.DataFrame(family_rows)
display(family_metrics.sort_values(["task","checkpoint","RMSE"],na_position="last").round(4))

### RT source-scale / coordinate-system diagnostic

The corpus contains multiple library-specific RT coordinate systems. The table and plots below compare each source's observed RT range with the unified model's prediction range, and quantify how much a **diagnostic-only per-source affine recalibration** can reduce RMSE. A large gain from affine recalibration together with high within-source correlation indicates scale/offset mismatch rather than failure to learn peptide retention ordering.

The affine recalibration shown here is **not** a proposed deployment correction and must not be fit on validation/test for training. It is only a diagnostic motivating train-only RT harmonization.


In [ ]:
rt_u = forward[(forward.checkpoint=="unified_v0134") & forward.target_rt.notna()].copy()

rt_range_rows=[]
rt_cal_rows=[]
for source,d in rt_u.groupby("source_id"):
    y=d.target_rt.to_numpy(float)
    p=d.predicted_rt.to_numpy(float)
    slope,intercept=np.polyfit(p,y,1)
    p_cal=intercept+slope*p
    raw_rmse=np.sqrt(np.mean((p-y)**2))
    affine_rmse=np.sqrt(np.mean((p_cal-y)**2))
    pearson=stats.pearsonr(y,p).statistic if np.std(y)>0 and np.std(p)>0 else np.nan
    spearman=stats.spearmanr(y,p).statistic if len(y)>1 else np.nan
    rt_range_rows.append({
        "source_id":source,
        "n":len(d),
        "target_min":np.min(y),
        "target_q01":np.quantile(y,0.01),
        "target_median":np.median(y),
        "target_q99":np.quantile(y,0.99),
        "target_max":np.max(y),
        "pred_min":np.min(p),
        "pred_median":np.median(p),
        "pred_max":np.max(p),
    })
    rt_cal_rows.append({
        "source_id":source,
        "n":len(d),
        "Pearson_r":pearson,
        "Spearman_rho":spearman,
        "raw_RMSE":raw_rmse,
        "affine_diag_RMSE":affine_rmse,
        "affine_slope_target_from_pred":slope,
        "affine_intercept_target_from_pred":intercept,
    })

rt_ranges=pd.DataFrame(rt_range_rows).sort_values("target_max")
rt_calibration=pd.DataFrame(rt_cal_rows).sort_values("raw_RMSE")
display(rt_ranges.round(3))
display(rt_calibration.round(4))

# How well does the unified model capture source-local ordering after removing
# each source's arbitrary location and scale?
z_rows=[]
percentile_mae=[]
for source,d in rt_u.groupby("source_id"):
    y=d.target_rt.to_numpy(float)
    p=d.predicted_rt.to_numpy(float)
    yz=(y-y.mean())/max(y.std(),1e-12)
    pz=(p-p.mean())/max(p.std(),1e-12)
    z_rows.append(pd.DataFrame({"source_id":source,"target_z":yz,"pred_z":pz}))
    y_rank=pd.Series(y).rank(pct=True).to_numpy()
    p_rank=pd.Series(p).rank(pct=True).to_numpy()
    percentile_mae.append(np.mean(np.abs(y_rank-p_rank)))
zdf=pd.concat(z_rows,ignore_index=True)
print(f"Source-centered/scaled pooled Pearson r: {np.corrcoef(zdf.target_z,zdf.pred_z)[0,1]:.4f}")
print(f"Source-centered/scaled pooled RMSE: {np.sqrt(np.mean((zdf.pred_z-zdf.target_z)**2)):.4f}")
print(f"Mean within-source percentile MAE: {np.mean(percentile_mae):.4f}")

# Observed target distributions expose incompatible source RT scales.
order=(rt_u.groupby("source_id").target_rt.median().sort_values().index.tolist())
fig,ax=plt.subplots(figsize=(11,max(6,0.38*len(order))))
data=[rt_u.loc[rt_u.source_id==s,"target_rt"].to_numpy() for s in order]
# Matplotlib-version-compatible boxplot labelling:
# avoid the deprecated/removed `labels=` / renamed `tick_labels=` keyword.
box_positions=np.arange(1,len(order)+1)
ax.boxplot(data,vert=False,positions=box_positions,showfliers=False)
ax.set_yticks(box_positions)
ax.set_yticklabels(order)
ax.set_title("Observed RT coordinate systems by source")
ax.set_xlabel("Source-native normalized RT / iRT-like value")
fig.tight_layout(); plt.show()

# Compare unified prediction and target robust ranges source by source.
range_plot=rt_ranges.set_index("source_id").loc[order]
y=np.arange(len(order))
fig,ax=plt.subplots(figsize=(11,max(6,0.38*len(order))))
ax.hlines(y, range_plot.target_q01, range_plot.target_q99, linewidth=4, alpha=0.55, label="Observed 1–99%")
ax.hlines(y, range_plot.pred_min, range_plot.pred_max, linewidth=2, alpha=0.8, label="Unified prediction min–max")
ax.scatter(range_plot.target_median, y, s=22, label="Observed median")
ax.scatter(range_plot.pred_median, y, s=22, marker="x", label="Predicted median")
ax.set_yticks(y); ax.set_yticklabels(order)
ax.set_xlabel("RT value")
ax.set_title("Observed versus unified RT dynamic range by source")
ax.legend()
fig.tight_layout(); plt.show()


### Cross-source RT label conflicts for identical peptidoforms

Because the intrinsic RT head is source-independent, identical peptidoform/charge pairs receive the same prediction. This table finds shared peptides whose source-native RT labels disagree most strongly. Large target spans with essentially zero prediction span are direct evidence that the source RT coordinates are not numerically interchangeable.


In [ ]:
shared=(rt_u.groupby(["peptidoform","charge"])
        .agg(sources=("source_id","nunique"),
             observations=("record_index","size"),
             target_min=("target_rt","min"),
             target_max=("target_rt","max"),
             predicted_min=("predicted_rt","min"),
             predicted_max=("predicted_rt","max"))
        .reset_index())
shared["target_span"]=shared.target_max-shared.target_min
shared["prediction_span"]=shared.predicted_max-shared.predicted_min
conflicts=(shared[shared.sources>=2]
           .sort_values("target_span",ascending=False)
           .head(25))
display(conflicts.round(4))

if len(conflicts):
    example=conflicts.iloc[0]
    detail=rt_u[(rt_u.peptidoform==example.peptidoform)&(rt_u.charge==example.charge)][
        ["source_id","target_rt","predicted_rt"]
    ].sort_values("target_rt")
    print(f"Most divergent shared example: {example.peptidoform} / charge {int(example.charge)}")
    display(detail.round(4))


## 3. v0.13.6 TRAIN-only cross-source RT harmonization

This section evaluates the v0.13.6 RT-target semantics experiment when its audit outputs are available. The source transforms were fit on **TRAIN only**; VALIDATION is used only for evaluation, and TEST remains unopened.

Two coordinate systems are shown:

- **latent harmonized RT** — the common source-independent target learned from TRAIN peptide overlaps;
- **source-native RT** — predictions transformed back into each library's original RT coordinate with the TRAIN-fit affine transform.


In [ ]:
if not rt_harmonization_available:
    display(Markdown(
        "**v0.13.6 RT harmonization outputs were not found.** "
        "Set `REDEEM_RT_HARMONIZATION_DIR` to the audit directory and re-execute the notebook."
    ))
else:
    init_global = rt_harm_initial_summary.loc[rt_harm_initial_summary.source_id=="__GLOBAL__"].iloc[0]
    final_global = rt_harm_final_summary.loc[rt_harm_final_summary.source_id=="__GLOBAL__"].iloc[0]

    summary = pd.DataFrame([
        {
            "checkpoint": "v0.13.4 parent / zero-step",
            "latent_MAE": init_global.latent_mae,
            "latent_RMSE": init_global.latent_rmse,
            "latent_Pearson": init_global.latent_pearson,
            "source_native_MAE": init_global.source_native_mae,
            "source_native_RMSE": init_global.source_native_rmse,
            "source_native_Pearson": init_global.source_native_pearson,
        },
        {
            "checkpoint": "v0.13.6 harmonized / step100",
            "latent_MAE": final_global.latent_mae,
            "latent_RMSE": final_global.latent_rmse,
            "latent_Pearson": final_global.latent_pearson,
            "source_native_MAE": final_global.source_native_mae,
            "source_native_RMSE": final_global.source_native_rmse,
            "source_native_Pearson": final_global.source_native_pearson,
        },
    ])
    display(summary.round(4))

    def pct_change(a, b):
        return 100.0 * (b / a - 1.0)

    print(
        "Latent RT RMSE change: "
        f"{pct_change(init_global.latent_rmse, final_global.latent_rmse):+.2f}%"
    )
    print(
        "Source-native RT RMSE change: "
        f"{pct_change(init_global.source_native_rmse, final_global.source_native_rmse):+.2f}%"
    )
    print(
        "Latent RT MAE change: "
        f"{pct_change(init_global.latent_mae, final_global.latent_mae):+.2f}%"
    )
    print(
        "Source-native RT MAE change: "
        f"{pct_change(init_global.source_native_mae, final_global.source_native_mae):+.2f}%"
    )

    # Latent RT: initial
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        rt_harm_initial.target_harmonized_rt,
        rt_harm_initial.predicted_harmonized_rt,
        s=8, alpha=0.25
    )
    lo = min(rt_harm_initial.target_harmonized_rt.min(), rt_harm_initial.predicted_harmonized_rt.min())
    hi = max(rt_harm_initial.target_harmonized_rt.max(), rt_harm_initial.predicted_harmonized_rt.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    ax.set_xlabel("Observed harmonized RT")
    ax.set_ylabel("Predicted harmonized RT")
    ax.set_title("v0.13.4 parent evaluated in harmonized RT coordinates")
    fig.tight_layout()
    plt.show()

    # Latent RT: final
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        rt_harm_final.target_harmonized_rt,
        rt_harm_final.predicted_harmonized_rt,
        s=8, alpha=0.25
    )
    lo = min(rt_harm_final.target_harmonized_rt.min(), rt_harm_final.predicted_harmonized_rt.min())
    hi = max(rt_harm_final.target_harmonized_rt.max(), rt_harm_final.predicted_harmonized_rt.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    ax.set_xlabel("Observed harmonized RT")
    ax.set_ylabel("Predicted harmonized RT")
    ax.set_title("v0.13.6 step100 in harmonized RT coordinates")
    fig.tight_layout()
    plt.show()

    # Source-native RT: initial
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        rt_harm_initial.target_source_rt,
        rt_harm_initial.predicted_source_rt,
        s=8, alpha=0.25
    )
    lo = min(rt_harm_initial.target_source_rt.min(), rt_harm_initial.predicted_source_rt.min())
    hi = max(rt_harm_initial.target_source_rt.max(), rt_harm_initial.predicted_source_rt.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    ax.set_xlabel("Observed source-native RT")
    ax.set_ylabel("Predicted source-native RT")
    ax.set_title("v0.13.4 parent transformed back to source-native RT")
    fig.tight_layout()
    plt.show()

    # Source-native RT: final
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(
        rt_harm_final.target_source_rt,
        rt_harm_final.predicted_source_rt,
        s=8, alpha=0.25
    )
    lo = min(rt_harm_final.target_source_rt.min(), rt_harm_final.predicted_source_rt.min())
    hi = max(rt_harm_final.target_source_rt.max(), rt_harm_final.predicted_source_rt.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
    ax.set_xlabel("Observed source-native RT")
    ax.set_ylabel("Predicted source-native RT")
    ax.set_title("v0.13.6 step100 transformed back to source-native RT")
    fig.tight_layout()
    plt.show()

    # Residual distributions
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(rt_harm_initial.source_error, bins=80, alpha=0.45, density=True, label="v0.13.4 parent")
    ax.hist(rt_harm_final.source_error, bins=80, alpha=0.45, density=True, label="v0.13.6 step100")
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("Source-native RT residual (prediction - observed)")
    ax.set_ylabel("Density")
    ax.set_title("Source-native RT residual distribution")
    ax.legend()
    fig.tight_layout()
    plt.show()

    # Per-source improvement.
    paired = (
        rt_harm_initial_summary[rt_harm_initial_summary.source_id!="__GLOBAL__"]
        .merge(
            rt_harm_final_summary[rt_harm_final_summary.source_id!="__GLOBAL__"],
            on="source_id", suffixes=("_initial", "_final")
        )
    )
    paired["rmse_change_pct"] = 100.0 * (
        paired.source_native_rmse_final / paired.source_native_rmse_initial - 1.0
    )
    paired["rmse_reduction_pct"] = -paired.rmse_change_pct
    display(
        paired[
            [
                "source_id",
                "n_initial",
                "source_native_rmse_initial",
                "source_native_rmse_final",
                "rmse_reduction_pct",
                "source_native_pearson_initial",
                "source_native_pearson_final",
            ]
        ]
        .sort_values("rmse_reduction_pct", ascending=False)
        .round(4)
    )

    plot_df = paired.sort_values("rmse_reduction_pct")
    fig, ax = plt.subplots(figsize=(11, max(6, 0.38*len(plot_df))))
    ax.barh(plot_df.source_id, plot_df.rmse_reduction_pct)
    ax.set_xlabel("Source-native RT RMSE reduction (%)")
    ax.set_title("v0.13.6 RT harmonization improvement by source")
    fig.tight_layout()
    plt.show()

    # Calibration coefficients, when available.
    if len(rt_harm_calibration):
        display(Markdown("### TRAIN-fit source calibration coefficients"))
        display(rt_harm_calibration.round(6))


In [ ]:
if rt_harmonization_available:
    fit_log = RT_HARMONIZATION_DIR / "fit.log"
    if fit_log.exists():
        text = fit_log.read_text(errors="replace")
        pattern = re.compile(
            r"rt_consistency\tlabel=(?P<label>[^\t]+)\t"
            r"shared_peptidoforms=(?P<shared>\d+)\t"
            r"observations=(?P<obs>\d+)\t"
            r"mae=(?P<mae>[-+0-9.eE]+)\t"
            r"rmse=(?P<rmse>[-+0-9.eE]+)"
        )
        rows = []
        for m in pattern.finditer(text):
            rows.append({
                "label": m.group("label"),
                "shared_peptidoforms": int(m.group("shared")),
                "observations": int(m.group("obs")),
                "MAE": float(m.group("mae")),
                "RMSE": float(m.group("rmse")),
            })
        if rows:
            consistency = pd.DataFrame(rows)
            display(Markdown("### TRAIN-fit harmonization consistency gate"))
            display(consistency.round(4))
        for key in ["fit_converged", "fit_iterations", "fitted_sources", "partition_identity_parity", "test_labels_consumed_by_fit"]:
            match = re.search(rf"^{key}\t(.+)$", text, flags=re.MULTILINE)
            if match:
                print(f"{key}: {match.group(1)}")


## 4. MS2 intensity prediction diagnostics

In [ ]:
for metric,label in [("ms2_cosine","Annotated-fragment cosine similarity"),("ms2_spectral_angle","Spectral angle"),("ms2_pearson","Fragment-intensity Pearson r")]:
    for checkpoint,d in forward.groupby("checkpoint"):
        vals=d[metric].dropna()
        fig,ax=plt.subplots(figsize=(7,5))
        ax.hist(vals,bins=40,alpha=0.8)
        ax.axvline(vals.median(),linestyle="--",label=f"median={vals.median():.3f}")
        ax.set_title(f"{label} — {checkpoint}")
        ax.set_xlabel(label); ax.set_ylabel("Peptides")
        ax.legend(); fig.tight_layout(); plt.show()

ms2_by_source=(forward.dropna(subset=["ms2_cosine"])
    .groupby(["checkpoint","source_id"])
    .agg(n=("record_index","size"),median_cosine=("ms2_cosine","median"),mean_cosine=("ms2_cosine","mean"),median_angle=("ms2_spectral_angle","median"))
    .reset_index())
display(ms2_by_source.sort_values(["checkpoint","median_cosine"],ascending=[True,False]).round(4))

### MS2 intensity sparsity / objective-shape diagnostic

MSE and spectral-shape metrics can move in opposite directions. This diagnostic quantifies prediction sparsity on **annotated observed fragments** and compares predicted-intensity distributions. A falling MSE accompanied by many exact zeros and lower cosine similarity indicates that the model is minimizing pointwise intensity error while losing relative spectral shape.


In [ ]:
ms2_sparsity=[]
for checkpoint,d in fragments.groupby("checkpoint"):
    t=d.target_intensity.to_numpy(float)
    p=d.predicted_intensity.to_numpy(float)
    ms2_sparsity.append({
        "checkpoint":checkpoint,
        "fragment_rows":len(d),
        "target_mean":np.mean(t),
        "prediction_mean":np.mean(p),
        "prediction_zero_fraction":np.mean(p==0),
        "fragment_MSE":np.mean((p-t)**2),
        "fragment_MAE":np.mean(np.abs(p-t)),
        "prediction_q50":np.quantile(p,0.50),
        "prediction_q90":np.quantile(p,0.90),
        "prediction_q99":np.quantile(p,0.99),
    })
ms2_sparsity=pd.DataFrame(ms2_sparsity).set_index("checkpoint")
display(ms2_sparsity.round(4))

fig,ax=plt.subplots(figsize=(8,5))
for checkpoint,d in fragments.groupby("checkpoint"):
    vals=d.predicted_intensity.to_numpy(float)
    ax.hist(vals,bins=60,alpha=0.45,density=True,label=checkpoint)
ax.set_xlim(left=0)
ax.set_xlabel("Predicted annotated-fragment intensity")
ax.set_ylabel("Density")
ax.set_title("Forward MS2 predicted-intensity distribution")
ax.legend()
fig.tight_layout(); plt.show()

# Paired per-peptide cosine change.
ms2_pair=(forward[forward.ms2_cosine.notna()]
          .pivot(index="record_index",columns="checkpoint",values="ms2_cosine")
          .dropna())
if {"baseline_v0129","unified_v0134"}.issubset(ms2_pair.columns):
    delta=ms2_pair["unified_v0134"]-ms2_pair["baseline_v0129"]
    print(f"Paired peptides: {len(delta):,}")
    print(f"Mean cosine change unified-baseline: {delta.mean():.4f}")
    print(f"Fraction with improved cosine: {(delta>0).mean():.3%}")


In [ ]:
def mirror_forward(record_index, checkpoint="unified_v0134"):
    d=fragments[(fragments.record_index==record_index)&(fragments.checkpoint==checkpoint)].dropna(subset=["product_mz"]).sort_values("product_mz")
    if len(d)==0:
        return
    fig,ax=plt.subplots(figsize=(12,5))
    observed_color="tab:blue"
    predicted_color="tab:orange"
    ax.vlines(d.product_mz,0,d.target_intensity,linewidth=1.2,color=observed_color,label="Observed")
    ax.vlines(d.product_mz,0,-d.predicted_intensity,linewidth=1.2,color=predicted_color,label="Predicted")
    ax.axhline(0,linewidth=0.8)
    for row in d.nlargest(min(8,len(d)),"target_intensity").itertuples():
        ax.text(row.product_mz,row.target_intensity+0.03,str(row.ion_label),rotation=90,ha="center",va="bottom",fontsize=8)
    info=forward[(forward.record_index==record_index)&(forward.checkpoint==checkpoint)].iloc[0]
    ax.set_title(f"Forward MS2 mirror: {info.peptidoform} | {info.source_id} | cosine={info.ms2_cosine:.3f}")
    ax.set_xlabel("m/z"); ax.set_ylabel("Observed (+) / predicted (-) intensity")
    from matplotlib.lines import Line2D
    handles=[Line2D([0],[0],color=observed_color,lw=2,label="Observed"),Line2D([0],[0],color=predicted_color,lw=2,label="Predicted")]
    ax.legend(handles=handles,loc="upper right")
    fig.tight_layout(); plt.show()

u=forward[(forward.checkpoint=="unified_v0134")&forward.ms2_cosine.notna()].copy()
if len(u):
    quantiles=[0.1,0.5,0.9]
    picked=[]
    for q in quantiles:
        target=u.ms2_cosine.quantile(q)
        idx=(u.ms2_cosine-target).abs().idxmin()
        picked.append(int(u.loc[idx,"record_index"]))
    for rid in dict.fromkeys(picked):
        mirror_forward(rid)

### v0.13.7 controlled spectral-shape objective audit

This optional section compares the **zero-step v0.13.6 parent** and the **v0.13.7 final continuation** on the exact same source-balanced **VALIDATION** records emitted by `foundation_benchmark_ms2_shape`. It reports raw annotated-fragment MSE/MAE separately from cosine/spectral-angle/Pearson so a shape gain cannot hide a calibration regression. It also tracks exact-zero predictions and mean predicted intensity, with source and source-family summaries.

The v0.13.7 first hypothesis deliberately keeps raw max-normalized intensity MSE and adds a per-spectrum cosine term. TEST remains unopened.


In [ ]:
if not ms2_shape_available:
    display(Markdown(
        "**v0.13.7 MS2 shape-audit outputs were not found.** "
        "Run `foundation_benchmark_ms2_shape` and set `REDEEM_MS2_SHAPE_AUDIT_DIR` if needed."
    ))
else:
    display(Markdown("#### Global parent vs v0.13.7 shape diagnostics"))
    global_ms2 = (ms2_shape_records
        .groupby("checkpoint")
        .agg(
            records=("record_index", "size"),
            pointwise_MSE=("pointwise_mse", "mean"),
            pointwise_MAE=("pointwise_mae", "mean"),
            mean_cosine=("cosine", "mean"),
            mean_spectral_angle=("spectral_angle", "mean"),
            mean_Pearson=("pearson", "mean"),
            mean_record_zero_fraction=("exact_zero_fraction", "mean"),
            mean_predicted_intensity=("mean_predicted_intensity", "mean"),
            mean_target_intensity=("mean_target_intensity", "mean"),
        ))
    # Exact fragment-weighted MSE/MAE/zero/intensity values come from the source summary.
    source_totals = []
    for checkpoint, d in ms2_shape_source_summary.groupby("checkpoint"):
        fragments_n = d.fragments.sum()
        source_totals.append({
            "checkpoint": checkpoint,
            "fragment_weighted_MSE": np.average(d.pointwise_mse, weights=d.fragments),
            "fragment_weighted_MAE": np.average(d.pointwise_mae, weights=d.fragments),
            "exact_zero_fraction": np.average(d.exact_zero_fraction, weights=d.fragments),
            "mean_predicted_intensity": np.average(d.mean_predicted_intensity, weights=d.fragments),
            "mean_target_intensity": np.average(d.mean_target_intensity, weights=d.fragments),
            "fragments": fragments_n,
        })
    source_totals = pd.DataFrame(source_totals).set_index("checkpoint")
    global_ms2 = global_ms2.drop(columns=["pointwise_MSE", "pointwise_MAE", "mean_record_zero_fraction", "mean_predicted_intensity", "mean_target_intensity"]).join(source_totals)
    display(global_ms2.round(4))
    display(Markdown("#### Paired per-spectrum changes"))
    display(ms2_shape_paired_summary.round(4))

    checkpoints = [c for c in ["initial_v0136", "final_v0137"] if c in global_ms2.index]
    if checkpoints:
        fig, ax = plt.subplots(figsize=(8, 5))
        plot_df = global_ms2.loc[checkpoints, ["mean_cosine", "mean_spectral_angle", "mean_Pearson"]]
        plot_df.plot(kind="bar", ax=ax)
        ax.set_ylabel("Similarity")
        ax.set_title("MS2 spectral-shape metrics — zero-step parent vs v0.13.7")
        ax.set_ylim(bottom=0)
        ax.legend(loc="best")
        fig.tight_layout(); plt.show()

        fig, ax = plt.subplots(figsize=(8, 5))
        plot_df = global_ms2.loc[checkpoints, ["fragment_weighted_MSE", "fragment_weighted_MAE", "exact_zero_fraction", "mean_predicted_intensity"]]
        plot_df.plot(kind="bar", ax=ax)
        ax.set_title("MS2 calibration / sparsity guards")
        ax.set_ylabel("Value")
        ax.legend(loc="best")
        fig.tight_layout(); plt.show()

    display(Markdown("#### Source-family summary"))
    family_view = ms2_shape_family_summary[[
        "checkpoint", "source_family", "records", "fragments",
        "pointwise_mse", "pointwise_mae", "mean_cosine", "mean_spectral_angle",
        "mean_pearson", "exact_zero_fraction", "mean_predicted_intensity",
        "mean_target_intensity",
    ]].copy()
    display(family_view.sort_values(["source_family", "checkpoint"]).round(4))

    family_cos = ms2_shape_family_summary.pivot(index="source_family", columns="checkpoint", values="mean_cosine")
    family_zero = ms2_shape_family_summary.pivot(index="source_family", columns="checkpoint", values="exact_zero_fraction")
    if {"initial_v0136", "final_v0137"}.issubset(family_cos.columns):
        family_delta = pd.DataFrame({
            "cosine_delta": family_cos["final_v0137"] - family_cos["initial_v0136"],
            "zero_fraction_delta": family_zero["final_v0137"] - family_zero["initial_v0136"],
        }).sort_values("cosine_delta")
        display(Markdown("#### Family-level v0.13.7 deltas"))
        display(family_delta.round(4))
        fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(family_delta))))
        family_delta[["cosine_delta", "zero_fraction_delta"]].plot(kind="barh", ax=ax)
        ax.axvline(0, linewidth=0.8)
        ax.set_title("v0.13.7 family deltas (positive cosine / negative zero fraction are favorable)")
        fig.tight_layout(); plt.show()


In [ ]:
if ms2_shape_available:
    def mirror_v0137(record_index, checkpoint="final_v0137"):
        d = ms2_shape_fragments[(ms2_shape_fragments.record_index == record_index) &
                                (ms2_shape_fragments.checkpoint == checkpoint)]
        d = d.dropna(subset=["product_mz"]).sort_values("product_mz")
        if len(d) == 0:
            return
        info = ms2_shape_records[(ms2_shape_records.record_index == record_index) &
                                 (ms2_shape_records.checkpoint == checkpoint)].iloc[0]
        fig, ax = plt.subplots(figsize=(12, 5))
        observed_color = "tab:blue"
        predicted_color = "tab:orange"
        ax.vlines(d.product_mz, 0, d.target_intensity, linewidth=1.2, color=observed_color, label="Observed")
        ax.vlines(d.product_mz, 0, -d.predicted_intensity, linewidth=1.2, color=predicted_color, label="Predicted")
        ax.axhline(0, linewidth=0.8)
        for row in d.nlargest(min(8, len(d)), "target_intensity").itertuples():
            ax.text(row.product_mz, row.target_intensity + 0.03, str(row.ion_label), rotation=90, ha="center", va="bottom", fontsize=8)
        ax.set_title(f"v0.13.7 MS2 mirror: {info.peptidoform} | {info.source_id} | cosine={info.cosine:.3f}")
        ax.set_xlabel("m/z"); ax.set_ylabel("Observed (+) / predicted (-) intensity")
        from matplotlib.lines import Line2D
        handles = [
            Line2D([0], [0], color=observed_color, lw=2, label="Observed"),
            Line2D([0], [0], color=predicted_color, lw=2, label="Predicted"),
        ]
        ax.legend(handles=handles, loc="upper right")
        fig.tight_layout(); plt.show()

    final_records = ms2_shape_records[(ms2_shape_records.checkpoint == "final_v0137") & ms2_shape_records.cosine.notna()]
    if len(final_records):
        picked = []
        for q in [0.1, 0.5, 0.9]:
            target = final_records.cosine.quantile(q)
            idx = (final_records.cosine - target).abs().idxmin()
            picked.append(int(final_records.loc[idx, "record_index"]))
        for record_index in dict.fromkeys(picked):
            mirror_v0137(record_index)


### v0.13.8 controlled MS2 output-activation rescue

This optional section compares three states on identical source-balanced **VALIDATION** records:

1. the accepted v0.13.7 checkpoint with historical hard ReLU;
2. the zero-step v0.13.8 checkpoint with the **same weights** and fixed Softplus `beta=5`;
3. the trained v0.13.8 Softplus continuation.

The accepted v0.13.7 objective remains `1.0 × raw masked MSE + 0.25 × (1 - cosine)`. Because Softplus is strictly positive, exact-zero fraction is retained only for ReLU compatibility; the primary sparsity diagnostics are now prediction fractions `<=1e-4`, `<=1e-3`, and `<=1e-2`, including channel-level summaries for the b/y output channels. TEST remains unopened.


In [ ]:
if not ms2_activation_available:
    display(Markdown(
        "**v0.13.8 MS2 activation-audit outputs were not found.** "
        "Run `foundation_benchmark_ms2_activation` and set `REDEEM_MS2_ACTIVATION_AUDIT_DIR` if needed."
    ))
else:
    display(Markdown("#### Three-state global activation diagnostics"))
    global_activation = []
    for checkpoint, d in ms2_activation_source_summary.groupby("checkpoint"):
        weights = d.fragments.to_numpy(float)
        global_activation.append({
            "checkpoint": checkpoint,
            "fragment_weighted_MSE": np.average(d.pointwise_mse, weights=weights),
            "fragment_weighted_MAE": np.average(d.pointwise_mae, weights=weights),
            "mean_cosine": np.average(d.mean_cosine, weights=d.records),
            "mean_spectral_angle": np.average(d.mean_spectral_angle, weights=d.records),
            "mean_Pearson": np.average(d.mean_pearson.fillna(0), weights=d.records),
            "exact_zero_fraction": np.average(d.exact_zero_fraction, weights=weights),
            "near_zero_1e4_fraction": np.average(d.near_zero_1e4_fraction, weights=weights),
            "near_zero_1e3_fraction": np.average(d.near_zero_1e3_fraction, weights=weights),
            "near_zero_1e2_fraction": np.average(d.near_zero_1e2_fraction, weights=weights),
            "mean_predicted_intensity": np.average(d.mean_predicted_intensity, weights=weights),
            "mean_target_intensity": np.average(d.mean_target_intensity, weights=weights),
            "fragments": int(weights.sum()),
        })
    global_activation = pd.DataFrame(global_activation).set_index("checkpoint")
    activation_order = [
        c for c in ["reference_relu_v0137", "initial_softplus_v0138", "final_softplus_v0138"]
        if c in global_activation.index
    ]
    display(global_activation.loc[activation_order].round(5))

    display(Markdown("#### Paired decomposition: immediate activation effect vs learning effect"))
    display(ms2_activation_paired_summary.set_index("comparison").round(5))

    if activation_order:
        fig, ax = plt.subplots(figsize=(9, 5))
        global_activation.loc[activation_order, ["mean_cosine", "mean_spectral_angle", "mean_Pearson"]].plot(kind="bar", ax=ax)
        ax.set_ylabel("Similarity")
        ax.set_title("MS2 shape: ReLU parent vs zero-step/trained Softplus")
        ax.set_ylim(bottom=0)
        fig.tight_layout(); plt.show()

        fig, ax = plt.subplots(figsize=(9, 5))
        global_activation.loc[activation_order, ["near_zero_1e4_fraction", "near_zero_1e3_fraction", "near_zero_1e2_fraction"]].plot(kind="bar", ax=ax)
        ax.set_ylabel("Annotated-fragment fraction")
        ax.set_title("MS2 near-zero predictions under activation rescue")
        ax.set_ylim(0, 1)
        fig.tight_layout(); plt.show()

    display(Markdown("#### Channel-level dead-zone audit"))
    channel_view = ms2_activation_channel_summary[ms2_activation_channel_summary.channel.isin([0, 1, 2, 3])].copy()
    display(channel_view.sort_values(["channel", "checkpoint"]).round(5))

    for metric, title in [
        ("near_zero_1e3_fraction", "Channel near-zero fraction (<=1e-3)"),
        ("mean_predicted_intensity", "Channel mean predicted annotated intensity"),
        ("pointwise_mse", "Channel pointwise MSE"),
    ]:
        pivot = channel_view.pivot(index="ion_family", columns="checkpoint", values=metric)
        cols = [c for c in activation_order if c in pivot.columns]
        if cols:
            fig, ax = plt.subplots(figsize=(9, 5))
            pivot[cols].plot(kind="bar", ax=ax)
            ax.set_title(title)
            ax.set_ylabel(metric)
            if "fraction" in metric:
                ax.set_ylim(0, 1)
            fig.tight_layout(); plt.show()

    display(Markdown("#### Source-family activation summary"))
    family_cols = [
        "checkpoint", "source_family", "records", "fragments", "pointwise_mse",
        "mean_cosine", "mean_spectral_angle", "mean_pearson",
        "near_zero_1e3_fraction", "near_zero_1e2_fraction",
        "mean_predicted_intensity", "mean_target_intensity",
    ]
    display(ms2_activation_family_summary[family_cols].sort_values(["source_family", "checkpoint"]).round(5))


In [ ]:
if ms2_activation_available:
    def mirror_v0138(record_index, checkpoint="final_softplus_v0138"):
        d = ms2_activation_fragments[(ms2_activation_fragments.record_index == record_index) &
                                     (ms2_activation_fragments.checkpoint == checkpoint)]
        d = d.dropna(subset=["product_mz"]).sort_values("product_mz")
        if len(d) == 0:
            return
        info = ms2_activation_records[(ms2_activation_records.record_index == record_index) &
                                      (ms2_activation_records.checkpoint == checkpoint)].iloc[0]
        fig, ax = plt.subplots(figsize=(12, 5))
        observed_color = "tab:blue"
        predicted_color = "tab:orange"
        ax.vlines(d.product_mz, 0, d.target_intensity, linewidth=1.2, color=observed_color, label="Observed")
        ax.vlines(d.product_mz, 0, -d.predicted_intensity, linewidth=1.2, color=predicted_color, label="Predicted")
        ax.axhline(0, linewidth=0.8)
        for row in d.nlargest(min(8, len(d)), "target_intensity").itertuples():
            ax.text(row.product_mz, row.target_intensity + 0.03, str(row.ion_label), rotation=90, ha="center", va="bottom", fontsize=8)
        ax.set_title(f"v0.13.8 MS2 mirror: {info.peptidoform} | {info.source_id} | {checkpoint} | cosine={info.cosine:.3f}")
        ax.set_xlabel("m/z"); ax.set_ylabel("Observed (+) / predicted (-) intensity")
        from matplotlib.lines import Line2D
        handles = [
            Line2D([0], [0], color=observed_color, lw=2, label="Observed"),
            Line2D([0], [0], color=predicted_color, lw=2, label="Predicted"),
        ]
        ax.legend(handles=handles, loc="upper right")
        fig.tight_layout(); plt.show()

    final_records = ms2_activation_records[(ms2_activation_records.checkpoint == "final_softplus_v0138") & ms2_activation_records.cosine.notna()]
    if len(final_records):
        picked = []
        for q in [0.1, 0.5, 0.9]:
            target = final_records.cosine.quantile(q)
            idx = (final_records.cosine - target).abs().idxmin()
            picked.append(int(final_records.loc[idx, "record_index"]))
        for record_index in dict.fromkeys(picked):
            mirror_v0138(record_index)


### v0.13.9 controlled b²/channel-1 output-row rescue

This optional section compares three states on identical source-balanced **VALIDATION** records:

1. the accepted v0.13.8 Softplus checkpoint;
2. the zero-step v0.13.9 checkpoint after resetting **only** the final MS2 `b²` / channel-1 weight row and bias to zero;
3. the trained v0.13.9 continuation.

The architecture and accepted objective remain unchanged: `1.0 × raw masked MSE + 0.25 × (1 - cosine)` with Softplus `beta=5`. The zero-row intervention makes the initial b² output `ln(2)/5 ≈ 0.1386` with derivative `0.5`, without fitting a validation statistic. The key question is whether training learns useful b² structure after the reset while preserving b¹/y¹/y², RT, CCS, and inverse/alignment guards. TEST remains unopened.


In [ ]:
if not ms2_b2_rescue_available:
    display(Markdown(
        "**v0.13.9 b²-rescue audit outputs were not found.** "
        "Run `foundation_benchmark_ms2_b2_rescue` and set `REDEEM_MS2_B2_RESCUE_AUDIT_DIR` if needed."
    ))
else:
    display(Markdown("#### Three-state global b²-rescue diagnostics"))
    global_b2 = []
    for checkpoint, d in ms2_b2_rescue_source_summary.groupby("checkpoint"):
        weights = d.fragments.to_numpy(float)
        global_b2.append({
            "checkpoint": checkpoint,
            "fragment_weighted_MSE": np.average(d.pointwise_mse, weights=weights),
            "fragment_weighted_MAE": np.average(d.pointwise_mae, weights=weights),
            "mean_cosine": np.average(d.mean_cosine, weights=d.records),
            "mean_spectral_angle": np.average(d.mean_spectral_angle, weights=d.records),
            "mean_Pearson": np.average(d.mean_pearson.fillna(0), weights=d.records),
            "near_zero_1e3_fraction": np.average(d.near_zero_1e3_fraction, weights=weights),
            "near_zero_1e2_fraction": np.average(d.near_zero_1e2_fraction, weights=weights),
            "mean_predicted_intensity": np.average(d.mean_predicted_intensity, weights=weights),
            "mean_target_intensity": np.average(d.mean_target_intensity, weights=weights),
            "fragments": int(weights.sum()),
        })
    global_b2 = pd.DataFrame(global_b2).set_index("checkpoint")
    b2_order = [
        c for c in ["reference_softplus_v0138", "initial_b2_reset_v0139", "final_b2_rescue_v0139"]
        if c in global_b2.index
    ]
    display(global_b2.loc[b2_order].round(5))

    display(Markdown("#### Paired decomposition: reset effect vs learned training effect"))
    display(ms2_b2_rescue_paired_summary.set_index("comparison").round(5))

    display(Markdown("#### Final-head parameter state"))
    display(ms2_b2_rescue_head_summary.sort_values(["channel", "checkpoint"]).round(7))

    channel_view = ms2_b2_rescue_channel_summary[
        ms2_b2_rescue_channel_summary.channel.isin([0, 1, 2, 3])
    ].copy()
    display(Markdown("#### Channel-level rescue audit"))
    display(channel_view.sort_values(["channel", "checkpoint"]).round(6))

    b2 = channel_view[channel_view.channel == 1].set_index("checkpoint")
    if len(b2):
        b2_cols = [c for c in b2_order if c in b2.index]
        display(Markdown("#### b²/channel-1 decision table"))
        display(b2.loc[b2_cols, [
            "fragments", "pointwise_mse", "pointwise_mae",
            "near_zero_1e4_fraction", "near_zero_1e3_fraction", "near_zero_1e2_fraction",
            "mean_target_intensity", "mean_predicted_intensity",
        ]].round(6))

    for metric, title in [
        ("near_zero_1e3_fraction", "Ion-channel near-zero fraction (<=1e-3)"),
        ("mean_predicted_intensity", "Ion-channel mean predicted annotated intensity"),
        ("pointwise_mse", "Ion-channel pointwise MSE"),
    ]:
        pivot = channel_view.pivot(index="ion_family", columns="checkpoint", values=metric)
        cols = [c for c in b2_order if c in pivot.columns]
        if cols:
            fig, ax = plt.subplots(figsize=(9, 5))
            pivot[cols].plot(kind="bar", ax=ax)
            ax.set_title(title)
            ax.set_ylabel(metric)
            if "fraction" in metric:
                ax.set_ylim(0, 1)
            fig.tight_layout(); plt.show()

    display(Markdown("#### Source-family b²-rescue summary"))
    family_cols = [
        "checkpoint", "source_family", "records", "fragments", "pointwise_mse",
        "mean_cosine", "mean_spectral_angle", "mean_pearson",
        "near_zero_1e3_fraction", "near_zero_1e2_fraction",
        "mean_predicted_intensity", "mean_target_intensity",
    ]
    display(ms2_b2_rescue_family_summary[family_cols].sort_values(["source_family", "checkpoint"]).round(5))


In [ ]:
if ms2_b2_rescue_available:
    def mirror_v0139(record_index, checkpoint="final_b2_rescue_v0139"):
        d = ms2_b2_rescue_fragments[(ms2_b2_rescue_fragments.record_index == record_index) &
                                     (ms2_b2_rescue_fragments.checkpoint == checkpoint)]
        d = d.dropna(subset=["product_mz"]).sort_values("product_mz")
        if len(d) == 0:
            return
        info = ms2_b2_rescue_records[(ms2_b2_rescue_records.record_index == record_index) &
                                      (ms2_b2_rescue_records.checkpoint == checkpoint)].iloc[0]
        fig, ax = plt.subplots(figsize=(12, 5))
        observed_color = "tab:blue"
        predicted_color = "tab:orange"
        ax.vlines(d.product_mz, 0, d.target_intensity, linewidth=1.2, color=observed_color, label="Observed")
        ax.vlines(d.product_mz, 0, -d.predicted_intensity, linewidth=1.2, color=predicted_color, label="Predicted")
        ax.axhline(0, linewidth=0.8)
        for row in d.nlargest(min(8, len(d)), "target_intensity").itertuples():
            ax.text(row.product_mz, row.target_intensity + 0.03, str(row.ion_label), rotation=90, ha="center", va="bottom", fontsize=8)
        ax.set_title(f"v0.13.9 MS2 mirror: {info.peptidoform} | {info.source_id} | {checkpoint} | cosine={info.cosine:.3f}")
        ax.set_xlabel("m/z"); ax.set_ylabel("Observed (+) / predicted (-) intensity")
        from matplotlib.lines import Line2D
        handles = [
            Line2D([0], [0], color=observed_color, lw=2, label="Observed"),
            Line2D([0], [0], color=predicted_color, lw=2, label="Predicted"),
        ]
        ax.legend(handles=handles, loc="upper right")
        fig.tight_layout(); plt.show()

    final_records = ms2_b2_rescue_records[(ms2_b2_rescue_records.checkpoint == "final_b2_rescue_v0139") & ms2_b2_rescue_records.cosine.notna()]
    if len(final_records):
        picked = []
        for q in [0.1, 0.5, 0.9]:
            target = final_records.cosine.quantile(q)
            idx = (final_records.cosine - target).abs().idxmin()
            picked.append(int(final_records.loc[idx, "record_index"]))
        for record_index in dict.fromkeys(picked):
            mirror_v0139(record_index)


## 5. Spectrum-to-peptide generation and identification benchmark

In [ ]:
prov_map=(provenance[["record_index","source_id","family"]].drop_duplicates("record_index") if len(provenance) else pd.DataFrame(columns=["record_index","source_id","family"]))

def attach_source(df):
    if "source_id" in df.columns:
        out=df.copy()
        out["family"]=out.source_id.map(source_to_family)
        return out
    out=df.merge(prov_map,on="record_index",how="left")
    if "family" not in out or out.family.isna().all():
        out["family"]="Unknown (provenance not bundled)"
    return out

baseline_inv=attach_source(baseline_inv)
unified_inv=attach_source(unified_inv)

def inverse_summary(df,label):
    mass=df[df.mass_valid==True].copy()
    records=pd.Index(df.record_index.unique())
    def pool_rate(col):
        hit=mass.groupby("record_index")[col].any().reindex(records,fill_value=False)
        return hit.mean()
    def top_rate(rank_col,col):
        top=mass[mass[rank_col]==1].drop_duplicates("record_index")
        hit=top.set_index("record_index")[col].reindex(records,fill_value=False)
        return hit.mean()
    return {
        "model":label,"records":len(records),
        "pool_peptidoform":pool_rate("peptidoform_exact"),
        "pool_sequence":pool_rate("sequence_exact"),
        "pool_IL":pool_rate("il_sequence_exact"),
        "fragment_top1_peptidoform":top_rate("fragment_mass_rank","peptidoform_exact"),
        "fragment_top1_IL":top_rate("fragment_mass_rank","il_sequence_exact"),
        "causal_top1_peptidoform":top_rate("causal_mass_rank","peptidoform_exact"),
        "causal_top1_IL":top_rate("causal_mass_rank","il_sequence_exact"),
        "locked_hybrid_top1_peptidoform":top_rate("fragment_causal_mass_rank","peptidoform_exact"),
        "locked_hybrid_top1_sequence":top_rate("fragment_causal_mass_rank","sequence_exact"),
        "locked_hybrid_top1_IL":top_rate("fragment_causal_mass_rank","il_sequence_exact"),
    }

inverse_metrics=pd.DataFrame([
    inverse_summary(baseline_inv,"pre-unified v0.12.x"),
    inverse_summary(unified_inv,"unified v0.13.4"),
])
display(inverse_metrics.set_index("model").round(4))

In [ ]:
def target_rank_table(df):
    valid=df[df.mass_valid==True]
    rows=[]
    for rank_col in ["fragment_mass_rank","causal_mass_rank","fragment_causal_mass_rank"]:
        exact=(valid[valid.peptidoform_exact==True].groupby("record_index")[rank_col].min())
        il=(valid[valid.il_sequence_exact==True].groupby("record_index")[rank_col].min())
        for kind,series in [("peptidoform",exact),("I/L",il)]:
            if len(series):
                rows.append({"ranking":rank_col,"match":kind,"n_hits":len(series),"median_rank":series.median(),"p90_rank":series.quantile(.9)})
    return pd.DataFrame(rows)

print("Pre-unified target ranks")
display(target_rank_table(baseline_inv))
print("Unified target ranks")
display(target_rank_table(unified_inv))

for label,df in [("pre-unified",baseline_inv),("unified",unified_inv)]:
    valid=df[(df.mass_valid==True)&(df.peptidoform_exact==True)]
    ranks=valid.groupby("record_index").fragment_causal_mass_rank.min()
    fig,ax=plt.subplots(figsize=(7,5))
    ax.hist(ranks,bins=np.arange(0.5,max(2,ranks.max()+1.5),1))
    ax.set_yscale("log")
    ax.set_xlabel("Best target rank under locked fragment + causal ranking")
    ax.set_ylabel("Records (log scale)")
    ax.set_title(f"Target rank distribution — {label}")
    fig.tight_layout(); plt.show()

In [ ]:
def inverse_by_family(df):
    mass=df[df.mass_valid==True]
    rows=[]
    for family,all_rows in df.groupby("family"):
        recs=pd.Index(all_rows.record_index.unique())
        m=mass[mass.family==family]
        top=m[m.fragment_causal_mass_rank==1].drop_duplicates("record_index")
        hit=top.set_index("record_index").peptidoform_exact.reindex(recs,fill_value=False)
        pool=m.groupby("record_index").peptidoform_exact.any().reindex(recs,fill_value=False)
        rows.append({"family":family,"records":len(recs),"pool_literal":pool.mean(),"locked_top1_literal":hit.mean()})
    return pd.DataFrame(rows)

print("Pre-unified by family")
display(inverse_by_family(baseline_inv).round(4))
print("Unified by family")
display(inverse_by_family(unified_inv).round(4))

## 6. Representative spectrum-to-peptide assignments

In [ ]:
AA_MASS={
'A':71.037113805,'R':156.10111105,'N':114.04292747,'D':115.026943065,'C':103.009184505,
'E':129.042593135,'Q':128.05857754,'G':57.021463735,'H':137.058911875,'I':113.084064015,
'L':113.084064015,'K':128.09496305,'M':131.040484645,'F':147.068413945,'P':97.052763875,
'S':87.032028435,'T':101.047678505,'W':186.07931298,'Y':163.063328575,'V':99.068413945,
}
PROTON=1.00727646677; WATER=18.010564684

def theoretical_by(sequence,max_charge=2):
    masses=[AA_MASS[a] for a in sequence]
    prefix=np.cumsum(masses)
    total=sum(masses)+WATER
    rows=[]
    for i in range(1,len(sequence)):
        bneutral=prefix[i-1]
        yneutral=total-prefix[i-1]
        for z in range(1,max_charge+1):
            rows.append((f"b{i}^{z}",(bneutral+z*PROTON)/z))
            rows.append((f"y{len(sequence)-i}^{z}",(yneutral+z*PROTON)/z))
    return rows

def assignment_mirror(record_index):
    top=unified_inv[(unified_inv.record_index==record_index)&(unified_inv.mass_valid==True)&(unified_inv.fragment_causal_mass_rank==1)]
    if len(top)==0: return
    row=top.iloc[0]
    if pd.notna(row.candidate_modifications) and str(row.candidate_modifications).strip(): return
    sp=spectra[spectra.record_index==record_index].sort_values("mz")
    if len(sp)==0: return
    theo=theoretical_by(str(row.candidate_sequence))
    fig,ax=plt.subplots(figsize=(13,5))
    observed_color="tab:blue"
    assigned_color="tab:orange"
    ax.vlines(sp.mz,0,sp.normalized_intensity,linewidth=0.8,color=observed_color,label="Observed spectrum")
    # Use nearest observed intensity for matched theoretical fragments, otherwise a short reference stick.
    neg=[]
    for label,mz in theo:
        ppm=np.abs(sp.mz.to_numpy()-mz)/mz*1e6
        if len(ppm) and ppm.min()<=20:
            intensity=float(sp.iloc[int(np.argmin(ppm))].normalized_intensity)
        else:
            intensity=0.08
        neg.append((label,mz,-intensity))
    for label,mz,intensity in neg:
        ax.vlines(mz,0,intensity,linewidth=0.8,color=assigned_color)
    ax.axhline(0,linewidth=0.8)
    for label,mz,intensity in sorted(neg,key=lambda x:x[2])[:8]:
        ax.text(mz,intensity-0.03,label,rotation=90,ha="center",va="top",fontsize=7)
    status="CORRECT" if bool(row.peptidoform_exact) else "INCORRECT"
    ax.set_title(f"Spectrum→peptide assignment ({status}): target={row.target_sequence}, assigned={row.candidate_sequence} | {row.source_id}")
    ax.set_xlabel("m/z"); ax.set_ylabel("Observed (+) / assigned theoretical (-)")
    from matplotlib.lines import Line2D
    handles=[Line2D([0],[0],color=observed_color,lw=2,label="Observed spectrum"),Line2D([0],[0],color=assigned_color,lw=2,label="Assigned theoretical fragments")]
    ax.legend(handles=handles,loc="upper right")
    fig.tight_layout(); plt.show()

u_top=unified_inv[(unified_inv.mass_valid==True)&(unified_inv.fragment_causal_mass_rank==1)].drop_duplicates("record_index")
u_top=u_top[(u_top.candidate_modifications.isna()) | (u_top.candidate_modifications.astype(str).str.len()==0)]
correct=u_top[u_top.peptidoform_exact==True].record_index.head(2).tolist()
incorrect=u_top[u_top.peptidoform_exact==False].record_index.head(2).tolist()
for rid in correct+incorrect:
    assignment_mirror(int(rid))

## Current accepted inverse-identification state: v0.13.23

This section reads the accepted v0.13.23 candidate TSV directly. It visualizes the gap between **candidate recall** and **top-1 ranking**, which motivated the rejected v0.14.0 setwise reranker. Set `REDEEM_V01323_CANDIDATES` if the default path differs.


The bounded **v0.14.1 contextual hierarchical residual reranker** preserves the frozen v0.13.23 score as a residual base, adds within-spectrum relative/rank and proposal-consensus features, and uses I/L plus exact hierarchical supervision. If its summary/diagnostics are present, the panels below compare its top-1 results and target-rank movement against the frozen ranker.


In [ ]:
V01323_CANDIDATES = Path(os.environ.get(
    'REDEEM_V01323_CANDIDATES',
    '/home/sing/Documents/datasets/redeem_foundation/bidirectional_mitm_v01323_final_selector/val128_v01323.tsv',
)).resolve()
V0140_SUMMARY = Path(os.environ.get(
    'REDEEM_V0140_SUMMARY',
    '/home/sing/Documents/datasets/redeem_foundation/setwise_reranker_v0140/model/validation_summary.tsv',
)).resolve()
V0141_SUMMARY = Path(os.environ.get(
    'REDEEM_V0141_SUMMARY',
    '/home/sing/Documents/datasets/redeem_foundation/setwise_reranker_v0141/model/validation_summary.tsv',
)).resolve()
V0141_DIAGNOSTICS = Path(os.environ.get(
    'REDEEM_V0141_DIAGNOSTICS',
    '/home/sing/Documents/datasets/redeem_foundation/setwise_reranker_v0141/model/validation_ranking_diagnostics.tsv',
)).resolve()

def bool_series(s):
    return s.astype(str).str.lower().isin(['true','1','yes'])

if V01323_CANDIDATES.exists():
    v13 = pd.read_csv(V01323_CANDIDATES, sep='\t')
    mass = v13[bool_series(v13.mass_valid)].copy()
    recs = pd.Index(v13.record_index.unique())
    oracle_literal = mass.groupby('record_index').peptidoform_exact.apply(bool_series).groupby(level=0).any().reindex(recs, fill_value=False)
    oracle_il = mass.groupby('record_index').il_sequence_exact.apply(bool_series).groupby(level=0).any().reindex(recs, fill_value=False)
    top = mass[mass.fragment_causal_mass_rank == 1].drop_duplicates('record_index').set_index('record_index')
    top_literal = bool_series(top.peptidoform_exact).reindex(recs, fill_value=False)
    top_il = bool_series(top.il_sequence_exact).reindex(recs, fill_value=False)
    current = pd.DataFrame([
        {'state':'v0.13.13 frozen', 'literal_oracle':28, 'IL_oracle':46, 'literal_top1':23, 'IL_top1':37},
        {'state':'v0.13.23 accepted', 'literal_oracle':int(oracle_literal.sum()), 'IL_oracle':int(oracle_il.sum()), 'literal_top1':int(top_literal.sum()), 'IL_top1':int(top_il.sum())},
    ])
    display(current)

    plot_df = current.set_index('state')[['literal_oracle','literal_top1','IL_oracle','IL_top1']]
    ax = plot_df.plot.bar(figsize=(10,5))
    ax.set_ylabel('Validation records / 128')
    ax.set_title('Candidate recall versus top-1 identification')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout(); plt.show()

    exact_ranks = mass[bool_series(mass.peptidoform_exact)].groupby('record_index').fragment_causal_mass_rank.min()
    il_ranks = mass[bool_series(mass.il_sequence_exact)].groupby('record_index').fragment_causal_mass_rank.min()
    fig, ax = plt.subplots(figsize=(9,5))
    bins = np.arange(0.5, max(3, int(max(exact_ranks.max(), il_ranks.max()))) + 1.5, 1)
    ax.hist([exact_ranks, il_ranks], bins=bins, label=['Literal target','I/L-equivalent target'], alpha=0.65)
    ax.set_yscale('log')
    ax.set_xlabel('Best target rank under fragment + 0.1 × N→C AR total')
    ax.set_ylabel('Records (log scale)')
    ax.set_title('Where the correct candidate sits inside the accepted v0.13.23 pool')
    ax.legend(); plt.tight_layout(); plt.show()

    missed = pd.DataFrame({
        'literal_oracle': oracle_literal, 'literal_top1': top_literal,
        'IL_oracle': oracle_il, 'IL_top1': top_il,
    })
    missed_literal = int((missed.literal_oracle & ~missed.literal_top1).sum())
    missed_il = int((missed.IL_oracle & ~missed.IL_top1).sum())
    display(pd.DataFrame([{'missed-but-present literal':missed_literal, 'missed-but-present I/L':missed_il}]))

    source_cols = ['from_diffusion','from_causal_beam','from_reverse_causal_beam','from_bidirectional_mitm']
    target_rows = mass[bool_series(mass.il_sequence_exact)].copy()
    source_counts = {c:int(bool_series(target_rows[c]).sum()) for c in source_cols if c in target_rows}
    if source_counts:
        ax = pd.Series(source_counts).rename(index={
            'from_diffusion':'Diffusion', 'from_causal_beam':'N→C causal',
            'from_reverse_causal_beam':'C→N causal', 'from_bidirectional_mitm':'MITM',
        }).plot.bar(figsize=(8,4))
        ax.set_ylabel('I/L-correct candidate rows')
        ax.set_title('Which proposal branches contribute correct candidates')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
        plt.tight_layout(); plt.show()
else:
    display(Markdown(f'**v0.13.23 candidate TSV not found:** `{V01323_CANDIDATES}`'))

if V0140_SUMMARY.exists():
    v14 = pd.read_csv(V0140_SUMMARY, sep='\t')
    display(Markdown('### v0.14.0 setwise reranker'))
    display(v14)
    if len(v14):
        row = v14.iloc[0]
        compare = pd.DataFrame({
            'Literal top-1':[row.get('legacy_top1_literal', np.nan), row.get('setwise_top1_literal', np.nan)],
            'I/L top-1':[row.get('legacy_top1_il', np.nan), row.get('setwise_top1_il', np.nan)],
        }, index=['Frozen ranker','v0.14.0 setwise'])
        ax=compare.plot.bar(figsize=(8,4))
        ax.set_ylabel('Validation records / 128')
        ax.set_title('Frozen ranker versus v0.14.0')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        plt.tight_layout(); plt.show()
else:
    display(Markdown('v0.14.0 summary is not present yet; this panel will populate automatically after training.'))


if V0141_SUMMARY.exists():
    v141 = pd.read_csv(V0141_SUMMARY, sep='\t')
    display(Markdown('### v0.14.1 contextual hierarchical residual reranker'))
    display(v141)
    if len(v141):
        row = v141.iloc[0]
        compare = pd.DataFrame({
            'Literal top-1':[
                row.get('legacy_top1_literal', np.nan),
                row.get('contextual_top1_literal', np.nan),
            ],
            'I/L top-1':[
                row.get('legacy_top1_il', np.nan),
                row.get('contextual_top1_il', np.nan),
            ],
        }, index=['Frozen v0.13.23 ranker','v0.14.1 contextual residual'])
        ax = compare.plot.bar(figsize=(9,4))
        ax.axhline(28, linestyle='--', linewidth=1, label='Literal gate = 28')
        ax.axhline(42, linestyle=':', linewidth=1, label='I/L gate = 42')
        ax.set_ylabel('Validation records / 128')
        ax.set_title('Frozen ranker versus v0.14.1')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.legend()
        plt.tight_layout(); plt.show()
else:
    display(Markdown('v0.14.1 summary is not present yet; this panel will populate automatically after training.'))

if V0141_DIAGNOSTICS.exists():
    d141 = pd.read_csv(V0141_DIAGNOSTICS, sep='\t', na_values=['NA'])
    for col in [
        'legacy_literal_rank','legacy_il_rank',
        'contextual_literal_rank','contextual_il_rank',
    ]:
        if col in d141:
            d141[col] = pd.to_numeric(d141[col], errors='coerce')

    literal = d141.dropna(subset=['legacy_literal_rank','contextual_literal_rank']).copy()
    il = d141.dropna(subset=['legacy_il_rank','contextual_il_rank']).copy()
    if len(literal) or len(il):
        movement = pd.DataFrame([
            {
                'target':'Literal',
                'improved':int((literal.contextual_literal_rank < literal.legacy_literal_rank).sum()),
                'unchanged':int((literal.contextual_literal_rank == literal.legacy_literal_rank).sum()),
                'worsened':int((literal.contextual_literal_rank > literal.legacy_literal_rank).sum()),
            },
            {
                'target':'I/L',
                'improved':int((il.contextual_il_rank < il.legacy_il_rank).sum()),
                'unchanged':int((il.contextual_il_rank == il.legacy_il_rank).sum()),
                'worsened':int((il.contextual_il_rank > il.legacy_il_rank).sum()),
            },
        ]).set_index('target')
        display(Markdown('#### Target-rank movement under v0.14.1'))
        display(movement)
        ax = movement.plot.bar(figsize=(8,4))
        ax.set_ylabel('Oracle-positive validation records')
        ax.set_title('Did the contextual residual move correct candidates up?')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        plt.tight_layout(); plt.show()

    if len(il):
        il_top = il[['legacy_il_rank','contextual_il_rank']].clip(upper=32)
        bins = np.arange(0.5, 33.5, 1)
        fig, ax = plt.subplots(figsize=(9,4))
        ax.hist(
            [il_top.legacy_il_rank, il_top.contextual_il_rank],
            bins=bins,
            label=['Frozen ranker','v0.14.1'],
            alpha=0.65,
        )
        ax.set_xlabel('Best I/L target rank (ranks >32 clipped to 32)')
        ax.set_ylabel('Validation records')
        ax.set_title('I/L target-rank distribution before and after v0.14.1')
        ax.legend()
        plt.tight_layout(); plt.show()
else:
    display(Markdown('v0.14.1 per-record ranking diagnostics are not present yet.'))


## 8. Final benchmark summary

In [ ]:
base=inverse_metrics.iloc[0]
uni=inverse_metrics.iloc[1]
rt_base=summary_global[(summary_global.checkpoint=="baseline_v0129")&(summary_global.task=="RT")].iloc[0]
rt_uni=summary_global[(summary_global.checkpoint=="unified_v0134")&(summary_global.task=="RT")].iloc[0]
ccs_base=summary_global[(summary_global.checkpoint=="baseline_v0129")&(summary_global.task=="CCS")].iloc[0]
ccs_uni=summary_global[(summary_global.checkpoint=="unified_v0134")&(summary_global.task=="CCS")].iloc[0]

text=f"""
### Validation conclusions

- **RT:** RMSE {rt_base.RMSE:.3f} → {rt_uni.RMSE:.3f} on the same expanded validation records.
- **CCS:** RMSE {ccs_base.RMSE:.3f} → {ccs_uni.RMSE:.3f} on the same expanded validation records.
- **MS2:** unified median annotated-fragment cosine = {ms2_global.loc['unified_v0134','median_cosine']:.3f}; inspect the sparsity diagnostic because pointwise MSE and spectral shape diverge.
- **Peptide candidate-pool literal recall:** {base.pool_peptidoform:.3%} → {uni.pool_peptidoform:.3%}.
- **Locked fragment + causal literal top-1:** {base.locked_hybrid_top1_peptidoform:.3%} → {uni.locked_hybrid_top1_peptidoform:.3%}.
- **Locked fragment + causal I/L top-1:** {base.locked_hybrid_top1_IL:.3%} → {uni.locked_hybrid_top1_IL:.3%}.

**RT interpretation:** source-native normalized RT coordinates are heterogeneous; pooled RMSE should be interpreted together with source-resolved correlation/range diagnostics.\n\nWhen available, the **v0.13.6 harmonized RT** section reports the TRAIN-only cross-source calibration and zero-step parent versus step100 adaptation.\n\nThese results are from **VALIDATION** only. Do not use the expanded TEST partition until model/data decisions are frozen.
"""
display(Markdown(text))